# Análise da Carteira — Previsão Jul/2026 vs. Realidade

Compara a **previsão de preço para Jul/2026** com a **cotação atual** (puxada ao vivo no `yfinance`) e com o **preço de compra em 01/06/2026**.

**O que este notebook faz:**
- **Acerto %** por ticker e geral → quão perto a previsão ficou do preço atual.
- **Lucro %** por ticker e geral → quanto teria rendido comprando em 01/06 e segurando até hoje.
- Exporta um **Excel interativo**: você muda a quantidade de ações e o Total Pago / Valor Atual / Lucro recalculam sozinhos.

**Premissa dos dados** (ajuste na seção 2 se estiver diferente):
- `previsao` = sua previsão para Jul/2026.
- `compra` = "o preço que tava" em 01/06/2026 (preço de compra).
- `preco_atual` = cotação de hoje, buscada no yfinance.

> Tickers da B3 usam o sufixo `.SA` no yfinance (ex.: `PETR4.SA`). Como a previsão é para julho e hoje ainda é junho, o "acerto" mostra o quanto a previsão está acompanhando o preço com ~1 mês de antecedência.

## 1. Instalação e imports

In [1]:
# Se faltar alguma biblioteca, rode uma vez (tire o # e execute):
# %pip install yfinance pandas openpyxl

import pandas as pd
import yfinance as yf
print("yfinance", yf.__version__)

c:\Users\Kaue Mandarino\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\Kaue Mandarino\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


yfinance 1.4.1


## 2. Dados de entrada (edite aqui)

In [2]:
# Previsão para Jul/2026 (R$)
previsao = {
    "PINE4": 16.18, "UGPA3": 30.88, "BHIA3": 2.12, "LIGT3": 3.93,
    "PETR4": 46.91, "CSAN3": 4.84, "PRIO3": 68.60, "RAIL3": 15.59,
    "ODPV3": 13.81, "SAUD3": 13.81, "SCAR3": 14.07, "AZZA3": 20.70,
}

# Preço de compra em 01/06/2026 (R$) -- "o preço que tava"
compra = {
    "PINE4": 12.64, "UGPA3": 24.92, "BHIA3": 1.26, "LIGT3": 2.69,
    "PETR4": 41.25, "CSAN3": 3.58, "PRIO3": 62.59, "RAIL3": 13.89,
    "ODPV3": 12.78, "SAUD3": 12.78, "SCAR3": 12.85, "AZZA3": 17.38,
}

DATA_COMPRA = "2026-06-01"   # usado só no item opcional (conferir compra no yfinance)
SUFIXO      = ".SA"          # sufixo da B3 no yfinance
QTD_PADRAO  = 1              # qtd de ações no Excel (você muda lá depois)

## 3. Cotação atual no yfinance

In [3]:
def cotacao_atual(tickers, sufixo=".SA"):
    """Retorna {ticker: ultimo_preco} usando yfinance (B3 -> sufixo .SA)."""
    precos = {}
    for t in tickers:
        p = None
        try:
            tk = yf.Ticker(t + sufixo)
            try:
                p = float(tk.fast_info["last_price"])
            except Exception:
                p = None
            if not p:                       # fallback: ultimo fechamento
                hist = tk.history(period="5d")
                if not hist.empty:
                    p = float(hist["Close"].dropna().iloc[-1])
        except Exception as e:
            print(f"  ! falha em {t}{sufixo}: {e}")
        precos[t] = p if p else float("nan")
        print(f"  {t:<6} -> {precos[t]}")
    return precos

print("Buscando cotacoes no yfinance...")
preco_atual = cotacao_atual(list(previsao.keys()), SUFIXO)

Buscando cotacoes no yfinance...
  PINE4  -> 12.0
  UGPA3  -> 25.600000381469727
  BHIA3  -> 1.1299999952316284
  LIGT3  -> 3.369999885559082
  PETR4  -> 38.060001373291016
  CSAN3  -> 3.759999990463257
  PRIO3  -> 53.290000915527344
  RAIL3  -> 13.6899995803833


$ODPV3.SA: possibly delisted; no price data found  (period=5d)


  ODPV3  -> 14.279999732971191
  SAUD3  -> 14.279999732971191
  SCAR3  -> 12.399999618530273
  AZZA3  -> 18.989999771118164


### 3b. (Opcional) Conferir o preço real de compra em 01/06
Se quiser validar o `compra` que você digitou, este trecho busca o fechamento real de 01/06/2026 no yfinance.

In [4]:
def preco_em_data(tickers, data, sufixo=".SA"):
    out = {}
    for t in tickers:
        try:
            h = yf.Ticker(t + sufixo).history(
                    start=data, end=str(pd.Timestamp(data) + pd.Timedelta(days=6)))
            out[t] = float(h["Close"].dropna().iloc[0]) if not h.empty else float("nan")
        except Exception:
            out[t] = float("nan")
    return out

# Descomente as 2 linhas abaixo para rodar:
# fechamento_real_01_06 = preco_em_data(list(previsao.keys()), DATA_COMPRA, SUFIXO)
# pd.Series(fechamento_real_01_06, name="Fech_01_06").round(2)

## 4. Acerto e lucro por ticker

In [5]:
df = pd.DataFrame({
    "Previsao_Jul26": previsao,
    "Preco_Compra":   compra,
    "Preco_Atual":    preco_atual,
})
df.index.name = "Ticker"

# Acerto da previsao (vs preco atual)
df["Erro_%"]   = (df["Previsao_Jul26"] - df["Preco_Atual"]) / df["Preco_Atual"] * 100
df["Acerto_%"] = (100 - df["Erro_%"].abs()).clip(lower=0)

# Lucro (comprou em 01/06, vale o preco atual)
df["Lucro_%"]        = (df["Preco_Atual"] - df["Preco_Compra"]) / df["Preco_Compra"] * 100
df["Lucro_R$_acao"]  =  df["Preco_Atual"] - df["Preco_Compra"]

df.round(2)

,Previsao_Jul26,Preco_Compra,Preco_Atual,Erro_%,Acerto_%,Lucro_%,Lucro_R$_acao
Ticker,,,,,,,
PINE4,16.18,12.64,12.00,34.83,65.17,-5.06,-0.64
UGPA3,30.88,24.92,25.60,20.62,79.38,2.73,0.68
BHIA3,2.12,1.26,1.13,87.61,12.39,-10.32,-0.13
LIGT3,3.93,2.69,3.37,16.62,83.38,25.28,0.68
PETR4,46.91,41.25,38.06,23.25,76.75,-7.73,-3.19
CSAN3,4.84,3.58,3.76,28.72,71.28,5.03,0.18
PRIO3,68.60,62.59,53.29,28.73,71.27,-14.86,-9.30
RAIL3,15.59,13.89,13.69,13.88,86.12,-1.44,-0.20
ODPV3,13.81,12.78,14.28,-3.29,96.71,11.74,1.50


## 5. Resultado geral

In [6]:
def brl(v):
    if pd.isna(v): return "-"
    return ("R$ " + f"{v:,.2f}").replace(",", "X").replace(".", ",").replace("X", ".")

acerto_geral = df["Acerto_%"].mean()
erro_medio   = df["Erro_%"].abs().mean()
total_pago   = df["Preco_Compra"].sum()      # 1 acao de cada
total_hoje   = df["Preco_Atual"].sum()
lucro_total  = total_hoje - total_pago
lucro_geral  = lucro_total / total_pago * 100

melhor = df["Lucro_%"].idxmax()
pior   = df["Lucro_%"].idxmin()

print("=" * 50)
print("RESULTADO GERAL  (comprando 1 acao de cada)")
print("=" * 50)
print(f"Acerto medio da previsao : {acerto_geral:5.1f}%   (erro medio {erro_medio:.1f}%)")
print(f"Total pago               : {brl(total_pago)}")
print(f"Valor hoje               : {brl(total_hoje)}")
print(f"Lucro / Prejuizo         : {brl(lucro_total)}  ({lucro_geral:+.1f}%)")
print("-" * 50)
print(f"Melhor papel: {melhor} ({df.loc[melhor,'Lucro_%']:+.1f}%)   "
      f"Pior papel: {pior} ({df.loc[pior,'Lucro_%']:+.1f}%)")

RESULTADO GERAL  (comprando 1 acao de cada)
Acerto medio da previsao :  76.4%   (erro medio 23.6%)
Total pago               : R$ 218,61
Valor hoje               : R$ 210,85
Lucro / Prejuizo         : R$ -7,76  (-3.5%)
--------------------------------------------------
Melhor papel: LIGT3 (+25.3%)   Pior papel: PRIO3 (-14.9%)


## 6. Exportar Excel interativo
Gera `carteira_analise.xlsx`. As colunas **Qtd Ações** e **Preço Atual** ficam editáveis — mude a quantidade e o Excel recalcula tudo sozinho.

In [7]:
def exportar_excel(dados, caminho="carteira_analise.xlsx", qtd_padrao=1):
    """
    Gera um Excel interativo da carteira.

    dados: lista de dicts, cada um com:
        ticker   (str)
        previsao (float)  -> preço previsto p/ Jul/2026
        compra   (float)  -> preço de compra em 01/06/2026
        atual    (float)  -> cotação atual (yfinance)

    As colunas 'Qtd Ações' e 'Preço Atual' ficam editáveis (em amarelo):
    ao mudar a quantidade ou o preço, TODO o resto recalcula sozinho via fórmulas.
    """
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

    FONTE = "Arial"
    F_RS  = '"R$" #,##0.00;("R$" #,##0.00);"-"'   # moeda, negativo entre (), zero = "-"
    F_PCT = '0.0%;(0.0%);"-"'
    F_QTD = '#,##0;(#,##0);"-"'

    azul        = Font(name=FONTE, color="0000FF")                 # entradas editáveis
    preto       = Font(name=FONTE, color="000000")                 # fórmulas/valores
    bold        = Font(name=FONTE, bold=True, color="000000")
    bold_branco = Font(name=FONTE, bold=True, color="FFFFFF")
    titulo_font = Font(name=FONTE, bold=True, size=14, color="FFFFFF")
    nota_font   = Font(name=FONTE, italic=True, size=9, color="595959")

    fill_titulo = PatternFill("solid", fgColor="2E75B6")
    fill_header = PatternFill("solid", fgColor="1F4E78")
    fill_total  = PatternFill("solid", fgColor="DDEBF7")
    fill_input  = PatternFill("solid", fgColor="FFF2CC")           # amarelo = editar aqui

    center = Alignment(horizontal="center", vertical="center")
    left   = Alignment(horizontal="left",   vertical="center")
    thin   = Side(style="thin", color="BFBFBF")
    borda  = Border(left=thin, right=thin, top=thin, bottom=thin)

    headers = ["Ticker", "Previsão Jul/26 (R$)", "Preço Compra 01/06 (R$)",
               "Preço Atual (R$)", "Qtd Ações", "Total Pago (R$)",
               "Valor Atual (R$)", "Lucro (R$)", "Lucro %", "Erro Prev. %", "Acerto %"]

    wb = Workbook()
    ws = wb.active
    ws.title = "Carteira"

    L_TIT, L_SUB, L_HEAD, PRIM = 1, 2, 3, 4
    n      = len(dados)
    ULT    = PRIM + n - 1
    L_TOT  = ULT + 1

    # título
    ws.merge_cells(start_row=L_TIT, start_column=1, end_row=L_TIT, end_column=11)
    c = ws.cell(L_TIT, 1, "Análise da Carteira — Previsão Jul/2026 vs. Hoje")
    c.font, c.fill, c.alignment = titulo_font, fill_titulo, center
    ws.row_dimensions[L_TIT].height = 26

    # subtítulo / instrução
    ws.merge_cells(start_row=L_SUB, start_column=1, end_row=L_SUB, end_column=11)
    c = ws.cell(L_SUB, 1, "Edite as colunas AMARELAS (Qtd Ações e Preço Atual) — o resto recalcula "
                          "sozinho.  Acerto % = quão perto a previsão ficou do preço atual.  "
                          "Lucro = preço atual vs. preço de compra.")
    c.font, c.alignment = nota_font, left
    ws.row_dimensions[L_SUB].height = 30

    # cabeçalho
    for j, h in enumerate(headers, start=1):
        c = ws.cell(L_HEAD, j, h)
        c.font, c.fill, c.alignment, c.border = bold_branco, fill_header, center, borda
    ws.row_dimensions[L_HEAD].height = 30

    # linhas de dados
    for i, d in enumerate(dados):
        r = PRIM + i
        ws.cell(r, 1, d["ticker"]).font = preto
        ws.cell(r, 2, d["previsao"]).font = preto
        ws.cell(r, 3, d["compra"]).font = preto
        ca = ws.cell(r, 4, d.get("atual")); ca.font = azul; ca.fill = fill_input   # EDITÁVEL
        cq = ws.cell(r, 5, qtd_padrao);     cq.font = azul; cq.fill = fill_input   # EDITÁVEL
        ws.cell(r, 6, f"=C{r}*E{r}").font = preto                       # total pago
        ws.cell(r, 7, f"=D{r}*E{r}").font = preto                       # valor atual
        ws.cell(r, 8, f"=G{r}-F{r}").font = preto                       # lucro R$
        ws.cell(r, 9,  f"=IF(C{r}=0,0,(D{r}-C{r})/C{r})").font = preto   # lucro %
        ws.cell(r, 10, f"=IF(D{r}=0,0,(B{r}-D{r})/D{r})").font = preto   # erro previsão %
        ws.cell(r, 11, f"=MAX(0,1-ABS(J{r}))").font = preto             # acerto %
        for col in (2, 3, 4, 6, 7, 8): ws.cell(r, col).number_format = F_RS
        ws.cell(r, 5).number_format = F_QTD
        for col in (9, 10, 11): ws.cell(r, col).number_format = F_PCT
        for col in range(1, 12):
            cc = ws.cell(r, col); cc.border = borda
            cc.alignment = left if col == 1 else center

    # linha de total / geral
    r = L_TOT
    ws.cell(r, 1, "TOTAL / GERAL").font = bold
    ws.cell(r, 5,  f"=SUM(E{PRIM}:E{ULT})").font = bold
    ws.cell(r, 6,  f"=SUM(F{PRIM}:F{ULT})").font = bold
    ws.cell(r, 7,  f"=SUM(G{PRIM}:G{ULT})").font = bold
    ws.cell(r, 8,  f"=SUM(H{PRIM}:H{ULT})").font = bold
    ws.cell(r, 9,  f"=IF(F{r}=0,0,H{r}/F{r})").font = bold             # lucro % geral = lucro/pago
    ws.cell(r, 10, f"=AVERAGE(J{PRIM}:J{ULT})").font = bold            # erro médio
    ws.cell(r, 11, f"=AVERAGE(K{PRIM}:K{ULT})").font = bold            # acerto médio
    for col in (6, 7, 8): ws.cell(r, col).number_format = F_RS
    ws.cell(r, 5).number_format = F_QTD
    for col in (9, 10, 11): ws.cell(r, col).number_format = F_PCT
    for col in range(1, 12):
        cc = ws.cell(r, col); cc.fill = fill_total; cc.border = borda
        cc.alignment = left if col == 1 else center

    larguras = {"A": 11, "B": 19, "C": 23, "D": 17, "E": 11,
                "F": 16, "G": 16, "H": 14, "I": 11, "J": 13, "K": 11}
    for col, w in larguras.items():
        ws.column_dimensions[col].width = w
    ws.freeze_panes = "B4"

    # aba Leia-me
    ws2 = wb.create_sheet("Leia-me")
    linhas = [
        ("Como usar esta planilha", True),
        ("", False),
        ("1) Quantas ações de cada papel: edite a coluna 'Qtd Ações' (amarelo).", False),
        ("   Total Pago, Valor Atual, Lucro (R$) e os totais recalculam automaticamente.", False),
        ("   Para a mesma quantidade em tudo, digite o número numa célula e arraste para baixo.", False),
        ("", False),
        ("2) 'Preço Atual' (amarelo) = cotação de hoje.", False),
        ("   O notebook preenche essa coluna sozinho via yfinance; também dá pra digitar/colar.", False),
        ("", False),
        ("Definições das fórmulas", True),
        ("- Lucro %        = (Preço Atual - Preço Compra) / Preço Compra", False),
        ("- Erro Prev. %   = (Previsão Jul/26 - Preço Atual) / Preço Atual   (quanto a previsão errou)", False),
        ("- Acerto %       = 1 - |Erro Prev. %|   (quanto acertou; nunca abaixo de 0)", False),
        ("- Lucro % geral  = Lucro total (R$) / Total pago (R$)", False),
        ("- Acerto % geral = média do Acerto % de cada ticker", False),
        ("", False),
        ("Preço de compra: data de 01/06/2026 (informado por você).", False),
    ]
    for i, (txt, b) in enumerate(linhas, start=1):
        cc = ws2.cell(i, 1, txt)
        cc.font = Font(name=FONTE, bold=b, size=12 if b else 10)
    ws2.column_dimensions["A"].width = 105

    wb.save(caminho)
    return caminho

In [8]:
dados = [{
    "ticker":   t,
    "previsao": float(previsao[t]),
    "compra":   float(compra[t]),
    "atual":    (None if pd.isna(preco_atual[t]) else float(preco_atual[t])),
} for t in previsao]

caminho = exportar_excel(dados, "carteira_analise.xlsx", qtd_padrao=QTD_PADRAO)
print("Excel salvo em:", caminho)

Excel salvo em: carteira_analise.xlsx
